In [4]:
import numpy as np
from scipy.spatial.distance import euclidean
import pandas as pd
import copy
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

import sys
import argparse
import glob
import cmlreaders as cml
import json
from matplotlib.ticker import FuncFormatter
import re
from scipy import signal
from scipy.stats import zscore
import mne
from mne import create_info, EpochsArray
warnings.filterwarnings('ignore')

# ============================================================================
# IMPORT IRASA (assumes you have it available)
# ============================================================================

import cmlreaders as cml
from irasa.IRASA import IRASA
try:
    from statsmodels.formula.api import mixedlm
    from statsmodels.stats.multitest import fdrcorrection
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q'])
    from statsmodels.formula.api import mixedlm
    from statsmodels.stats.multitest import fdrcorrection
import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from scipy.spatial.distance import euclidean
import copy
import cmlreaders as cml

whole_df = cml.CMLReader.get_data_index()
exp = 'DBOY1'
subjects = whole_df.query('experiment == @exp')['subject'].unique()
subject = subjects[0]

sub_df = whole_df.query('experiment == @exp and subject == @subject')
session = sub_df['session'].unique()[0]

reader = cml.CMLReader(subject, exp, session=session)
evs = reader.load('task_events')

print(f"Subject: {subject}, Session: {session}")
print(f"Shape: {evs.shape}")
print(evs.head(20))

Subject: R1494D, Session: 0
Shape: (428, 28)
    eegoffset  correctPointingDirection                         eegfile  \
0      882107                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
1      884123                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
2      886040                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
3      888173                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
4      890173                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
5      892174                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
6      894122                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
7      896207                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
8      898172                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
9      900206                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
10     902123                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
11     904256                 -999.0000  R1494D_DBOY1_0

In [5]:
import cmlreaders as cml
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)

whole_df = cml.CMLReader.get_data_index()
exp = 'DBOY1'
subjects = whole_df.query('experiment == @exp')['subject'].unique()

print(f"Found {len(subjects)} subjects")

Found 44 subjects


In [12]:
import cmlreaders as cml
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

pd.set_option('display.max_rows', None)

# ============================================================================
# LOAD IRASA CSV
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_clean.csv')

# ============================================================================
# FILTER TO GAMMA BAND AND BUILD LONG FORMAT
# ============================================================================
gamma     = df[(df['freq_hz'] >= 4) & (df['freq_hz'] <= 8)].copy()
gamma     = gamma[gamma['SP_clustering_from_prev'].notna()].copy()
gamma_avg = gamma.copy()
gamma     = df[df['freq_hz'] >= 40].copy()


sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

tc = gamma.copy()
tc['clustering_score'] = tc['T_clustering_from_prev']
tc['clustering_type']  = 'T'

long = pd.concat([sp, tc], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RHP'] = (long['region'] == 'RHP').astype(int)
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean_map          = gamma_avg.groupby('subject')[col].mean()
    long[col + '_between'] = long['subject'].map(subj_mean_map)
    long[col + '_within']  = long[col] - long[col + '_between']

# ============================================================================
# PARALLEL WORKER — one subject × session
# ============================================================================

def process_session(subject, session, exp):
    import cmlreaders as cml
    import gc

    recall_rows = []
    nav_rows    = []

    try:
        reader = cml.CMLReader(subject, exp, session=session)
        evs    = reader.load('task_events').reset_index(drop=True)

        # ── GET SAMPLING RATE ─────────────────────────────────────────────
        sr = 1000  # default
        try:
            one_event = evs[evs['type'] == 'WORD'].iloc[[0]]
            eeg_small = reader.load('eeg', events=one_event, rel_start=0, rel_stop=1)
            sr        = float(eeg_small.samplerate)
            del eeg_small
            gc.collect()
        except Exception:
            sr = 1000
            print(f"  ! {subject} sess {session}: sr fallback to 1000")

        # ── LOOP OVER TRIALS ──────────────────────────────────────────────
        for trial, trial_evs in evs.groupby('trial'):
            if trial < 0:
                continue

            trial_evs = trial_evs.reset_index(drop=True)

            word_evs = trial_evs[trial_evs['type'] == 'WORD']
            n_words  = len(word_evs)
            if n_words == 0:
                continue

            rec_words   = trial_evs[trial_evs['type'] == 'REC_WORD']
            n_correct   = len(rec_words[rec_words['itemno'] > 0])
            n_intrusion = len(rec_words[rec_words['itemno'] < 0])

            recall_rows.append({
                'subject':     subject,
                'session':     session,
                'trial':       trial,
                'experiment':  exp,
                'n_words':     n_words,
                'n_correct':   n_correct,
                'n_intrusion': n_intrusion,
                'recall_rate': n_correct / n_words,
                'samplerate':  sr,
            })

            for i, row in trial_evs.iterrows():
                if row['type'] == 'pointing finished':
                    subsequent = trial_evs[trial_evs.index > i]
                    next_word  = subsequent[subsequent['type'] == 'WORD']

                    if len(next_word) > 0:
                        next_word_row    = next_word.iloc[0]
                        nav_time_samples = next_word_row['eegoffset'] - row['eegoffset']
                        nav_time_seconds = nav_time_samples / sr

                        nav_rows.append({
                            'subject':                 subject,
                            'session':                 session,
                            'trial':                   trial,
                            'experiment':              exp,
                            'samplerate':              sr,
                            'navigation_time_samples': nav_time_samples,
                            'navigation_time_seconds': nav_time_seconds,
                        })

    except Exception as e:
        print(f"  ✗ {subject} sess {session} ({exp}): {e}")

    return recall_rows, nav_rows
# ============================================================================
# BUILD TASK LIST
# ============================================================================

whole_df       = cml.CMLReader.get_data_index()
EXPERIMENTS    = ['DBOY1', 'EFRCourierReadOnly']
irasa_subjects = set(long['subject'].unique())

tasks = []
for exp in EXPERIMENTS:
    all_exp_subs = set(whole_df.query('experiment == @exp')['subject'].unique())
    subjects     = sorted(irasa_subjects & all_exp_subs)
    for subject in subjects:
        sub_df   = whole_df.query('experiment == @exp and subject == @subject')
        sessions = sub_df['session'].unique()
        for session in sessions:
            tasks.append((subject, session, exp))

print(f"Total subject-session tasks: {len(tasks)}")

# ============================================================================
# RUN IN PARALLEL
# ============================================================================

N_JOBS = 12  # conservative to avoid OOM on rhino2

print(f"Running on {N_JOBS} cores...")

results = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(process_session)(subject, session, exp)
    for subject, session, exp in tasks
)

# ============================================================================
# COLLECT RESULTS
# ============================================================================

all_recall_rows = []
all_nav_rows    = []

for recall_rows, nav_rows in results:
    all_recall_rows.extend(recall_rows)
    all_nav_rows.extend(nav_rows)

recall_df = pd.DataFrame(all_recall_rows)
nav_df    = pd.DataFrame(all_nav_rows)

print(f"\nrecall_df shape: {recall_df.shape}")
print(f"nav_df shape:    {nav_df.shape}")
print(f"Experiments in recall_df: {recall_df['experiment'].unique().tolist()}")
print(f"Subjects in recall_df:    {sorted(recall_df['subject'].unique())}")

# QC — flag suspiciously long navigation times
nav_df['nav_too_long'] = nav_df['navigation_time_seconds'] > 180
print(f"\nNav events > 3 min: {nav_df['nav_too_long'].sum()}")
if nav_df['nav_too_long'].sum() > 0:
    print(nav_df[nav_df['nav_too_long']][
        ['subject', 'session', 'trial', 'navigation_time_seconds', 'samplerate']
    ].to_string(index=False))

# Sampling rate summary per subject
print("\nSampling rate per subject:")
print(nav_df.groupby('subject')['samplerate'].unique().to_string())

# ============================================================================
# TRIAL-LEVEL AGGREGATION
# ============================================================================

# Nav time — mean across pointing events per trial
nav_trial = (
    nav_df.groupby(['subject', 'session', 'trial'])
    .agg(
        nav_time_seconds = ('navigation_time_seconds', 'mean'),
        samplerate       = ('samplerate',              'first'),
        n_nav_events     = ('navigation_time_seconds', 'count'),
    )
    .reset_index()
)

# Recall rate per trial
trial_recall = (
    recall_df[['subject', 'session', 'trial', 'recall_rate',
               'n_correct', 'n_words', 'n_intrusion']]
    .drop_duplicates(subset=['subject', 'session', 'trial'])
)

# Gamma — mean across frequencies and words within trial, per region
gamma_trial = (
    gamma_avg
    .groupby(['subject', 'session', 'trial', 'region'])
    .agg(
        encoding_frac_ssl  = ('encoding_frac_ssl',  'mean'),
        encoding_osc_ssl   = ('encoding_osc_ssl',   'mean'),
        retrieval_frac_ssl = ('retrieval_frac_ssl',  'mean'),
        retrieval_osc_ssl  = ('retrieval_osc_ssl',   'mean'),
    )
    .reset_index()
)

gamma_trial_wide = gamma_trial.pivot_table(
    index   = ['subject', 'session', 'trial'],
    columns = 'region',
    values  = ['encoding_frac_ssl', 'encoding_osc_ssl',
               'retrieval_frac_ssl', 'retrieval_osc_ssl'],
).reset_index()

gamma_trial_wide.columns = [
    '_'.join(filter(None, map(str, col))).strip()
    if col[1] != '' else col[0]
    for col in gamma_trial_wide.columns
]

# SP and TC clustering — mean across recalled words per trial per region
clust_trial = (
    long[long['clust_T'] == 0]
    .groupby(['subject', 'session', 'trial', 'region'])
    .agg(sp_clustering = ('clustering_score', 'mean'))
    .reset_index()
    .merge(
        long[long['clust_T'] == 1]
        .groupby(['subject', 'session', 'trial', 'region'])
        .agg(tc_clustering = ('clustering_score', 'mean'))
        .reset_index(),
        on  = ['subject', 'session', 'trial', 'region'],
        how = 'outer'
    )
)

clust_trial_wide = clust_trial.pivot_table(
    index   = ['subject', 'session', 'trial'],
    columns = 'region',
    values  = ['sp_clustering', 'tc_clustering'],
).reset_index()

clust_trial_wide.columns = [
    '_'.join(filter(None, map(str, col))).strip()
    if col[1] != '' else col[0]
    for col in clust_trial_wide.columns
]

# ============================================================================
# MERGE ALL INTO ONE TRIAL-LEVEL CSV
# ============================================================================

trial_df = (
    gamma_trial_wide
    .merge(clust_trial_wide, on=['subject', 'session', 'trial'], how='outer')
    .merge(trial_recall,     on=['subject', 'session', 'trial'], how='left')
    .merge(nav_trial,        on=['subject', 'session', 'trial'], how='left')
    .sort_values(['subject', 'session', 'trial'])
    .reset_index(drop=True)
)

# ============================================================================
# SANITY CHECK
# ============================================================================

print(f"\ntrial_df shape: {trial_df.shape}")
print(f"Subjects: {sorted(trial_df['subject'].unique())}")
print(f"\nColumn list:")
for c in trial_df.columns:
    n_nan = trial_df[c].isna().sum()
    print(f"  {c:<45} NaNs: {n_nan}")

print(f"\nSample rows:")
print(trial_df.head(10).to_string(index=False))

print(f"\nNav time (seconds) summary:")
print(trial_df['nav_time_seconds'].describe())

print(f"\nRecall rate summary:")
print(trial_df['recall_rate'].describe())

# ============================================================================
# SAVE
# ============================================================================

trial_df.to_csv('trial_level_summary.csv', index=False)
print("\nSaved: trial_level_summary.csv")
print(f"Shape: {trial_df.shape}")
print(f"Columns: {trial_df.columns.tolist()}")

Total subject-session tasks: 84
Running on 12 cores...


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  1.1min
[Parallel(n_jobs=12)]: Done  78 out of  84 | elapsed:  1.7min remaining:    7.9s
[Parallel(n_jobs=12)]: Done  84 out of  84 | elapsed:  1.8min finished



recall_df shape: (278, 9)
nav_df shape:    (3334, 7)
Experiments in recall_df: ['DBOY1', 'EFRCourierReadOnly']
Subjects in recall_df:    ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1520T', 'R1521E', 'R1523E', 'R1529T', 'R1530J', 'R1532T', 'R1536J', 'R1537T', 'R1538E', 'R1539E', 'R1542J', 'R1543E', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1561E', 'R1564J', 'R1567T', 'R1569T', 'R1570T', 'R1571T', 'R1572T', 'R1573T', 'R1575E', 'R1620J', 'R1637T', 'R1642J', 'R1651J', 'R1653J']

Nav events > 3 min: 2
subject  session  trial  navigation_time_seconds  samplerate
 R1509E        0      1               260.813477      1024.0
 R1538E        1      0               363.142090      2048.0

Sampling rate per subject:
subject
R1494D            [1000.0]
R1501J            [2000.0]
R1502D            [1000.0]
R1503E            [2048.0]
R1504E            [2048.0]
R1509E            [1024.0]
R1520T            [1000.0]
R1521E            [2048.0]
R1523E            [2048.0]
R1529T          

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# ============================================================================
# SUBJECT-LEVEL BEHAVIORAL AGGREGATES
# ============================================================================

recall_subj = (
    recall_df.groupby('subject')
    .agg(mean_recall_rate = ('recall_rate', 'mean'))
    .reset_index()
)

nav_clean = nav_df[nav_df['navigation_time_seconds'] <= 180].copy()
nav_subj = (
    nav_clean.groupby('subject')
    .agg(
        mean_nav_time_seconds = ('navigation_time_seconds', 'mean'),  # seconds only
        n_nav_events          = ('navigation_time_seconds', 'count'),
    )
    .reset_index()
)

# Fixed: aggregate from full long, not clust_T==0 filtered
clust_subj = (
    long.groupby('subject')
    .agg(
        mean_sp_clustering = ('SP_clustering_from_prev', 'mean'),
        mean_tc_clustering = ('T_clustering_from_prev',  'mean'),
    )
    .reset_index()
)

subj_behav = (
    clust_subj
    .merge(recall_subj, on='subject', how='inner')
    .merge(nav_subj,    on='subject', how='inner')
)

# ============================================================================
# SUBJECT-LEVEL NEURAL AGGREGATES — split by region
# ============================================================================

neural_cols = [
    'encoding_frac_ssl', 'encoding_osc_ssl',
    'retrieval_frac_ssl', 'retrieval_osc_ssl',
]

subj_neural_rows = []

for subj_id, grp in gamma_avg.groupby('subject'):
    row = {'subject': subj_id}
    for col in neural_cols:
        for region, rgrp in grp.groupby('region'):
            tag = 'LHP' if region == 'LHP' else 'RHP'
            row[f'{col}_{tag}'] = rgrp[col].mean()
    subj_neural_rows.append(row)

subj_neural = pd.DataFrame(subj_neural_rows)

# ============================================================================
# MERGE BEHAVIORAL + NEURAL
# ============================================================================

subj = subj_behav.merge(subj_neural, on='subject', how='inner')
subj = subj.loc[:, ~subj.columns.duplicated()]

# Fixed: dropna on mean_nav_time_seconds, not mean_nav_time_samples
subj = subj.dropna(subset=['mean_sp_clustering', 'mean_tc_clustering',
                            'mean_recall_rate',   'mean_nav_time_seconds'])

print(f"Subjects with all variables: {len(subj)}")
print(f"Columns: {subj.columns.tolist()}")

# ============================================================================
# VARIABLE DEFINITIONS FOR MATRIX
# ============================================================================

var_defs = {
    'Nav time (s)':  'mean_nav_time_seconds',
    'SP clustering': 'mean_sp_clustering',
    'TC clustering': 'mean_tc_clustering',
    'Recall rate':   'mean_recall_rate',
    'Enc frac LHP':  'encoding_frac_ssl_LHP',
    'Enc frac RHP':  'encoding_frac_ssl_RHP',
    'Enc osc LHP':   'encoding_osc_ssl_LHP',
    'Enc osc RHP':   'encoding_osc_ssl_RHP',
    'Ret frac LHP':  'retrieval_frac_ssl_LHP',
    'Ret frac RHP':  'retrieval_frac_ssl_RHP',
    'Ret osc LHP':   'retrieval_osc_ssl_LHP',
    'Ret osc RHP':   'retrieval_osc_ssl_RHP',
}

# Keep only variables whose columns actually exist in subj
var_defs = {k: v for k, v in var_defs.items() if v in subj.columns}
missing  = [v for v in var_defs.values() if v not in subj.columns]
if missing:
    print(f"WARNING — missing columns dropped: {missing}")

labels = list(var_defs.keys())
cols   = list(var_defs.values())
n_vars = len(labels)

# ============================================================================
# CORRELATION MATRIX + p-VALUES
# ============================================================================

corr_mat = np.full((n_vars, n_vars), np.nan)
p_mat    = np.full((n_vars, n_vars), np.nan)

for i, ci in enumerate(cols):
    for j, cj in enumerate(cols):
        xi = subj[ci].squeeze()
        xj = subj[cj].squeeze()
        valid_mask = xi.notna() & xj.notna()
        xi_vals = xi[valid_mask].values.astype(float)
        xj_vals = xj[valid_mask].values.astype(float)
        if len(xi_vals) < 4:
            continue
        if xi_vals.std() < 1e-10 or xj_vals.std() < 1e-10:
            continue
        r, p = stats.pearsonr(xi_vals, xj_vals)
        corr_mat[i, j] = r
        p_mat[i, j]    = p

# ============================================================================
# PRINT CORRELATION TABLE
# ============================================================================

print("\nPearson correlations (r / p):")
print(f"{'':20s}" + "".join(f"{l:>14s}" for l in labels))
for i, li in enumerate(labels):
    row_str = f"{li:20s}"
    for j in range(n_vars):
        if j <= i:
            row_str += f"{'—':>14s}"
        else:
            r = corr_mat[i, j]
            p = p_mat[i, j]
            if np.isnan(r):
                row_str += f"{'NaN':>14s}"
            else:
                stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
                row_str += f"{r:+.2f}{stars:3s}{'':5s}"
    print(row_str)

# ============================================================================
# FIGURE — full correlation matrix heatmap
# ============================================================================

group_colors = {
    'Nav time (s)':  '#555555',
    'SP clustering': '#2166AC',
    'TC clustering': '#D6604D',
    'Recall rate':   '#1A9850',
    'Enc frac LHP':  '#7B2D8B',
    'Enc frac RHP':  '#B15CC8',
    'Enc osc LHP':   '#C07020',
    'Enc osc RHP':   '#E8A040',
    'Ret frac LHP':  '#7B2D8B',
    'Ret frac RHP':  '#B15CC8',
    'Ret osc LHP':   '#C07020',
    'Ret osc RHP':   '#E8A040',
}

fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor('white')

im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')

# Annotate cells
for i in range(n_vars):
    for j in range(n_vars):
        r = corr_mat[i, j]
        p = p_mat[i, j]
        if np.isnan(r):
            continue
        stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        text_color = 'white' if abs(r) > 0.55 else 'black'
        ax.text(j, i, f'{r:+.2f}\n{stars}',
                ha='center', va='center', fontsize=7.5,
                color=text_color, fontweight='500')

# Axis tick labels colored by group
ax.set_xticks(range(n_vars))
ax.set_yticks(range(n_vars))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for tick, label in zip(ax.get_xticklabels(), labels):
    tick.set_color(group_colors.get(label, 'black'))
for tick, label in zip(ax.get_yticklabels(), labels):
    tick.set_color(group_colors.get(label, 'black'))

# White divider lines separating variable groups
n_behav = sum(1 for l in labels if l in ['Nav time (s)','SP clustering','TC clustering','Recall rate'])
n_enc   = sum(1 for l in labels if l.startswith('Enc'))
b1 = n_behav - 0.5
b2 = n_behav + n_enc - 0.5
for boundary in [b1, b2]:
    ax.axhline(boundary, color='white', linewidth=2.5)
    ax.axvline(boundary, color='white', linewidth=2.5)

# Group labels on top via twin axis
group_spans = [
    ('Behavioral',  0,               n_behav - 1),
    ('Encoding γ',  n_behav,         n_behav + n_enc - 1),
    ('Retrieval γ', n_behav + n_enc, n_vars - 1),
]
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks([(s + e) / 2 for _, s, e in group_spans])
ax2.set_xticklabels([g for g, _, _ in group_spans], fontsize=10, fontweight='500')
ax2.tick_params(length=0)

plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label='Pearson r')
ax.set_title(
    f'Correlation matrix — behavioral & gamma-band iEEG  (n={len(subj)})\n'
    f'* p<.05   ** p<.01   *** p<.001  (uncorrected)',
    fontsize=11, fontweight='500', pad=28
)
plt.tight_layout()
plt.savefig('full_correlation_matrix.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: full_correlation_matrix.png")

Subjects with all variables: 36
Columns: ['subject', 'mean_sp_clustering', 'mean_tc_clustering', 'mean_recall_rate', 'mean_nav_time_seconds', 'n_nav_events', 'encoding_frac_ssl_RHP', 'encoding_osc_ssl_RHP', 'retrieval_frac_ssl_RHP', 'retrieval_osc_ssl_RHP', 'encoding_frac_ssl_LHP', 'encoding_osc_ssl_LHP', 'retrieval_frac_ssl_LHP', 'retrieval_osc_ssl_LHP']

Pearson correlations (r / p):
                      Nav time (s) SP clustering TC clustering   Recall rate  Enc frac LHP  Enc frac RHP   Enc osc LHP   Enc osc RHP  Ret frac LHP  Ret frac RHP   Ret osc LHP   Ret osc RHP
Nav time (s)                     —+0.07        -0.30        -0.26        -0.02        -0.02        -0.30        -0.16        -0.09        -0.08        +0.12        +0.07        
SP clustering                    —             —-0.01        -0.08        +0.11        +0.27        -0.46*       -0.46*       +0.07        +0.20        -0.03        -0.20        
TC clustering                    —             —             —-0.

In [14]:
"""
Subject-level analysis: average everything per subject first,
then run simple OLS (one data point per participant).

4 models:
  A: LHP gamma ~ SP_clustering + nav_speed + n_correct
  B: RHP gamma ~ SP_clustering + nav_speed + n_correct
  C: LHP gamma ~ TC_clustering + nav_speed + n_correct
  D: RHP gamma ~ TC_clustering + nav_speed + n_correct

FDR correction across all 12 predictor tests.
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')
print(f"Trial-level shape: {df.shape}")
print(f"Subjects: {sorted(df['subject'].unique())}")

cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_time_seconds',
]
cols = [c for c in cols if c in df.columns]

# Average all trials within each subject (one row per subject)
subj = df.groupby('subject')[cols].mean().reset_index()

# Navigation speed = 1 / nav_time
subj['nav_speed'] = 1.0 / subj['nav_time_seconds']

print(f"\nSubject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols + ['nav_speed']].describe().round(3).to_string())

# ============================================================================
# 2. Z-SCORE EVERYTHING
# ============================================================================

to_z = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_speed',
]
to_z = [c for c in to_z if c in subj.columns]

for col in to_z:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 3. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':     label,
            'outcome':   outcome,
            'predictor': pred,
            'beta':      fit.params[pred],
            'SE':        fit.bse[pred],
            't':         fit.tvalues[pred],
            'p':         fit.pvalues[pred],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 4. RUN MODELS
# ============================================================================

res_A = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['sp_clustering_LHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'A: LHP gamma ~ SP + nav_speed + n_correct',
    data       = subj,
)

res_B = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['sp_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'B: RHP gamma ~ SP + nav_speed + n_correct',
    data       = subj,
)

res_C = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['tc_clustering_LHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'C: LHP gamma ~ TC + nav_speed + n_correct',
    data       = subj,
)

res_D = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['tc_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'D: RHP gamma ~ TC + nav_speed + n_correct',
    data       = subj,
)

# ============================================================================
# 5. COMBINED TABLE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

# FDR across all 12 tests (3 predictors x 4 models)
_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("ALL RESULTS — BH-FDR corrected across 12 tests")
print("="*70)
print(results[['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
              ].to_string(index=False))

# ============================================================================
# 6. SAVE
# ============================================================================

results.to_csv('gamma_osc_subject_level_results.csv', index=False)
subj.to_csv('subject_level_averages.csv', index=False)
print("\nSaved: gamma_osc_subject_level_results.csv")
print("Saved: subject_level_averages.csv")

Trial-level shape: (217, 22)
Subjects: ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1520T', 'R1521E', 'R1523E', 'R1529T', 'R1530J', 'R1532T', 'R1536J', 'R1537T', 'R1538E', 'R1539E', 'R1542J', 'R1543E', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1561E', 'R1564J', 'R1567T', 'R1569T', 'R1570T', 'R1571T', 'R1572T', 'R1573T', 'R1575E', 'R1620J', 'R1637T', 'R1642J', 'R1651J', 'R1653J']

Subject-level shape: (36, 10)

Descriptives:
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP  n_correct  nav_time_seconds  nav_speed
count                27.000                27.000             27.000             27.000             27.000             27.000     36.000            36.000     36.000
mean                  0.301                 0.295              0.514              0.529              0.595              0.604      5.134            31.509      0.034
std                   0.504                 0.605       

In [15]:
"""
Plots matching the reference style:
  Panel A  — bar plot: beta ± SE for SP clustering models (LHP + RHP, 3 predictors)
  Panel B  — bar plot: beta ± SE for TC clustering models (LHP + RHP, 3 predictors)
  Panel C  — scatter: LHP & RHP overlaid, SP clustering vs hippocampal power
  Panel D  — scatter: LHP & RHP overlaid, TC clustering vs hippocampal power

One figure produced per component × stage combination:
  osc_encoding, osc_retrieval, frac_encoding, frac_retrieval  → 4 figures total
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# ============================================================================
# 1. LOAD DATA
# ============================================================================

df      = pd.read_csv('trial_level_summary.csv')
results = pd.read_csv('gamma_frac_osc_enc_ret_subject_level_results.csv')

# ── Rebuild subject-level averages ──────────────────────────────────────────
neural_cols = [
    'encoding_osc_ssl_LHP',  'encoding_osc_ssl_RHP',
    'encoding_frac_ssl_LHP', 'encoding_frac_ssl_RHP',
    'retrieval_osc_ssl_LHP', 'retrieval_osc_ssl_RHP',
    'retrieval_frac_ssl_LHP','retrieval_frac_ssl_RHP',
]
behav_cols = [
    'sp_clustering_LHP', 'sp_clustering_RHP',
    'tc_clustering_LHP', 'tc_clustering_RHP',
    'n_correct',         'nav_time_seconds',
]
use_cols = [c for c in neural_cols + behav_cols if c in df.columns]
subj = df.groupby('subject')[use_cols].mean().reset_index()
subj['nav_speed'] = 1.0 / subj['nav_time_seconds']

to_z = [c for c in use_cols if c != 'nav_time_seconds'] + ['nav_speed']
for col in to_z:
    if col in subj.columns:
        subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 2. STYLE CONSTANTS  (matching reference exactly)
# ============================================================================

BLUE  = '#185FA5'
GREEN = '#0F6E56'
GRAY  = '#888780'
RED   = '#D85A30'

plt.rcParams.update({
    'font.family':        'sans-serif',
    'font.size':          11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':     0.8,
    'xtick.major.size':   3,
    'ytick.major.size':   3,
})

PRED_DISPLAY = {
    'sp_clustering_LHP_z': 'SP clust',
    'sp_clustering_RHP_z': 'SP clust',
    'tc_clustering_LHP_z': 'TC clust',
    'tc_clustering_RHP_z': 'TC clust',
    'nav_speed_z':         'Nav speed',
    'n_correct_z':         'N correct',
}

# ============================================================================
# 3. HELPER: extract stats from results table
# ============================================================================

def get_stats(label_exact, clust_pred):
    """Return list of dicts {beta, SE, p_fdr} for [clust, nav_speed, n_correct]."""
    sub  = results[results['label'] == label_exact]
    rows = []
    for pred in [clust_pred, 'nav_speed_z', 'n_correct_z']:
        row = sub[sub['predictor'] == pred]
        if row.empty:
            rows.append({'beta': 0.0, 'SE': 0.0, 'p_fdr': 1.0})
        else:
            rows.append(row.iloc[0][['beta', 'SE', 'p_fdr']].to_dict())
    return rows

# ============================================================================
# 4. PANEL A/B: grouped bar plot  (reference style)
# ============================================================================

def plot_betas(ax, lhp_label, rhp_label, clust_pred_lhp, clust_pred_rhp,
               clust_display, title):
    """
    Grouped bar: 3 x-positions (clustering, nav_speed, n_correct),
    2 bars each (LHP=BLUE, RHP=GREEN).
    """
    x      = np.arange(3)
    width  = 0.35
    off    = [-width / 2, width / 2]
    specs  = [
        (lhp_label, clust_pred_lhp, 'LHP', BLUE),
        (rhp_label, clust_pred_rhp, 'RHP', GREEN),
    ]

    all_betas, all_ses = [], []
    for idx, (label, clust_pred, hem, color) in enumerate(specs):
        rows  = get_stats(label, clust_pred)
        betas = np.array([r['beta']  for r in rows])
        ses   = np.array([r['SE']    for r in rows])
        ps    = np.array([r['p_fdr'] for r in rows])
        all_betas.extend(betas); all_ses.extend(ses)

        ax.bar(x + off[idx], betas, width,
               color=color, alpha=0.85, label=hem,
               zorder=3, linewidth=0)
        ax.errorbar(x + off[idx], betas, yerr=ses,
                    fmt='none', color='#444',
                    linewidth=1.2, capsize=3, zorder=4)

        for i, (b, se, p) in enumerate(zip(betas, ses, ps)):
            if p < 0.05:
                marker = '***' if p < .001 else ('**' if p < .01 else '*')
                ypos = b + se + 0.05 if b >= 0 else b - se - 0.13
                ax.text(x[i] + off[idx], ypos, marker,
                        ha='center', va='bottom', fontsize=10,
                        color=RED, fontweight='bold')
            elif p < 0.1:
                ypos = b + se + 0.05 if b >= 0 else b - se - 0.13
                ax.text(x[i] + off[idx], ypos, '†',
                        ha='center', va='bottom', fontsize=11, color=GRAY)

    ax.axhline(0, color='#aaa', linewidth=0.8, zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels([clust_display, 'Nav speed', 'N correct'], fontsize=10)
    ax.set_ylabel('Standardized β', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='normal', pad=8)
    ax.legend(fontsize=9, frameon=False)

    # y-limits matching reference approach
    ymin = min(min(all_betas) - max(all_ses) - 0.25, -0.3)
    ymax = max(max(all_betas) + max(all_ses) + 0.25,  0.7)
    ax.set_ylim(ymin, ymax)
    ax.yaxis.grid(True, linewidth=0.4, color='#ddd', zorder=0)
    ax.set_axisbelow(True)

# ============================================================================
# 5. PANEL C/D: scatter with both hemispheres overlaid
# ============================================================================

def plot_scatter(ax, outcome_lhp, outcome_rhp, pred_lhp, pred_rhp,
                 xlabel, ylabel, title, solid_lhp=True):
    """
    Both hemispheres on the same axes.
    LHP = BLUE (solid line), RHP = GREEN (solid line).
    """
    for outcome, pred, color, hem, ls in [
        (outcome_lhp, pred_lhp, BLUE,  'LHP', '-'),
        (outcome_rhp, pred_rhp, GREEN, 'RHP', '-'),
    ]:
        sub = subj[[pred, outcome]].dropna()
        if sub.empty:
            continue
        x_ = sub[pred].values
        y_ = sub[outcome].values

        slope, intercept, r, p, _ = stats.linregress(x_, y_)
        xline = np.linspace(x_.min(), x_.max(), 100)

        ax.scatter(x_, y_, color=color, alpha=0.70, s=40, zorder=3,
                   edgecolors='white', linewidths=0.5, label=hem)
        ax.plot(xline, slope * xline + intercept,
                color=color, linewidth=1.8, zorder=4, linestyle=ls)

        # CI band
        n   = len(x_)
        res = y_ - (slope * x_ + intercept)
        se_ = np.std(res, ddof=2)
        ci  = se_ * stats.t.ppf(0.975, df=n - 2) * np.sqrt(
                  1/n + (xline - x_.mean())**2 / np.sum((x_ - x_.mean())**2))
        ax.fill_between(xline,
                        slope * xline + intercept - ci,
                        slope * xline + intercept + ci,
                        color=color, alpha=0.10, zorder=2)

        p_str = f'p = {p:.3f}' if p >= 0.001 else 'p < 0.001'
        # stack r/p annotations vertically by hemisphere
        y_ann = 0.93 if hem == 'LHP' else 0.82
        ax.text(0.05, y_ann, f'{hem}  r = {r:.2f},  {p_str}',
                transform=ax.transAxes,
                fontsize=8.5, color=color, va='top')

    ax.axhline(0, color='#ccc', lw=0.6, zorder=1)
    ax.axvline(0, color='#ccc', lw=0.6, zorder=1)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='normal', pad=8)
    ax.legend(fontsize=9, frameon=False)
    ax.yaxis.grid(True, linewidth=0.4, color='#ddd', zorder=0)
    ax.set_axisbelow(True)

# ============================================================================
# 6. MAIN LOOP — one figure per component × stage
# ============================================================================

COMBOS = [
    ('osc',  'encoding'),
    ('osc',  'retrieval'),
    ('frac', 'encoding'),
    ('frac', 'retrieval'),
]

COMP_DISPLAY  = {'osc': 'Oscillatory', 'frac': 'Fractal'}
STAGE_DISPLAY = {'encoding': 'Encoding',  'retrieval': 'Retrieval'}

for comp, stage in COMBOS:

    comp_d  = COMP_DISPLAY[comp]
    stage_d = STAGE_DISPLAY[stage]

    # ── label strings (must match results CSV exactly) ────────────────────
    lhp_sp = f'LHP {comp} ({stage}) ~ SP_clustering + nav_speed + n_correct'
    rhp_sp = f'RHP {comp} ({stage}) ~ SP_clustering + nav_speed + n_correct'
    lhp_tc = f'LHP {comp} ({stage}) ~ TC_clustering + nav_speed + n_correct'
    rhp_tc = f'RHP {comp} ({stage}) ~ TC_clustering + nav_speed + n_correct'

    # ── neural outcome column names ───────────────────────────────────────
    out_lhp = f'{stage}_{comp}_ssl_LHP_z'
    out_rhp = f'{stage}_{comp}_ssl_RHP_z'
    power_lbl = f'{comp_d} power ({stage_d}, z)'

    # ── build figure ──────────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.48, wspace=0.35)

    fig.suptitle(
        f'Hippocampal {comp_d} Power ({stage_d}) — Subject-level OLS',
        fontsize=13, fontweight='bold', y=1.01
    )

    # ── Panel A: SP beta plot ─────────────────────────────────────────────
    ax_a = fig.add_subplot(gs[0, 0])
    plot_betas(
        ax_a,
        lhp_label      = lhp_sp,
        rhp_label      = rhp_sp,
        clust_pred_lhp = 'sp_clustering_LHP_z',
        clust_pred_rhp = 'sp_clustering_RHP_z',
        clust_display  = 'SP clust',
        title          = 'A   SP clustering models — beta coefficients',
    )

    # ── Panel B: TC beta plot ─────────────────────────────────────────────
    ax_b = fig.add_subplot(gs[0, 1])
    plot_betas(
        ax_b,
        lhp_label      = lhp_tc,
        rhp_label      = rhp_tc,
        clust_pred_lhp = 'tc_clustering_LHP_z',
        clust_pred_rhp = 'tc_clustering_RHP_z',
        clust_display  = 'TC clust',
        title          = 'B   TC clustering models — beta coefficients',
    )

    # ── Panel C: SP scatter (LHP + RHP overlaid) ─────────────────────────
    ax_c = fig.add_subplot(gs[1, 0])
    plot_scatter(
        ax_c,
        outcome_lhp = out_lhp,
        outcome_rhp = out_rhp,
        pred_lhp    = 'sp_clustering_LHP_z',
        pred_rhp    = 'sp_clustering_RHP_z',
        xlabel      = 'SP clustering (z)',
        ylabel      = power_lbl,
        title       = 'C   SP clustering vs hippocampal power',
    )

    # ── Panel D: TC scatter (LHP + RHP overlaid) ─────────────────────────
    ax_d = fig.add_subplot(gs[1, 1])
    plot_scatter(
        ax_d,
        outcome_lhp = out_lhp,
        outcome_rhp = out_rhp,
        pred_lhp    = 'tc_clustering_LHP_z',
        pred_rhp    = 'tc_clustering_RHP_z',
        xlabel      = 'TC clustering (z)',
        ylabel      = power_lbl,
        title       = 'D   TC clustering vs hippocampal power',
    )

    # ── footnote ──────────────────────────────────────────────────────────
    fig.text(
        0.5, 0.01,
        '* p < .05   ** p < .01   *** p < .001 (FDR-corrected)   '
        '† p < .10 trend   Error bars = ±1 SE',
        ha='center', fontsize=8, color='#888'
    )

    fname = f'figure_{comp}_{stage}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"Saved: {fname}")

print("\nAll figures saved.")

Saved: figure_osc_encoding.png
Saved: figure_osc_retrieval.png
Saved: figure_frac_encoding.png
Saved: figure_frac_retrieval.png

All figures saved.


In [8]:
"""
Trial-level LMM replacing the subject-level OLS.

Random effects structure:
  (1 | subject) + (1 | subject/session)
  implemented via Mundlak-style vc_formula in statsmodels:
    groups = subject
    vc_formula = {'subj_sess': '0 + C(subj_sess)'}

4 models:
  A: LHP gamma ~ SP_clustering + nav_speed + n_correct + (1|subj) + (1|subj/sess)
  B: RHP gamma ~ SP_clustering + nav_speed + n_correct + (1|subj) + (1|subj/sess)
  C: LHP gamma ~ TC_clustering + nav_speed + n_correct + (1|subj) + (1|subj/sess)
  D: RHP gamma ~ TC_clustering + nav_speed + n_correct + (1|subj) + (1|subj/sess)

FDR correction across all 12 predictor tests (3 predictors × 4 models).
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD DATA  (trial level — no averaging)
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')
print(f"Trial-level shape: {df.shape}")
print(f"Subjects:  {df['subject'].nunique()}")
print(f"Sessions:  {df.groupby('subject')['session'].nunique().to_dict()}")

# ── create subject×session grouping variable ─────────────────────────────────
df['subj_sess'] = df['subject'] + '_' + df['session'].astype(str)
print(f"Subject×session units: {df['subj_sess'].nunique()}")

# ── keep only needed columns ─────────────────────────────────────────────────
cols_needed = [
    'subject', 'session', 'subj_sess',
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_time_seconds',
]
cols_needed = [c for c in cols_needed if c in df.columns]
df = df[cols_needed].copy()

# navigation speed
df['nav_speed'] = 1.0 / df['nav_time_seconds']

# ============================================================================
# 2. Z-SCORE  (across all trials, grand mean / SD)
# ============================================================================

to_z = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_speed',
]
to_z = [c for c in to_z if c in df.columns]

for col in to_z:
    df[col + '_z'] = (df[col] - df[col].mean()) / df[col].std()

print(f"\nDescriptives (z-scored):")
z_cols = [c + '_z' for c in to_z]
print(df[z_cols].describe().round(3).to_string())

# ============================================================================
# 3. LMM HELPER
# ============================================================================

def run_lmm(outcome, main_pred, label, data):
    """
    Fit LMM:
      outcome ~ main_pred + nav_speed_z + n_correct_z
               + (1 | subject) + (1 | subject/session)

    statsmodels implementation:
      groups  = subject           → subject-level random intercept
      vc_formula = {'subj_sess': '0 + C(subj_sess)'}
                                  → session-within-subject random intercept
    """
    predictors = [main_pred, 'nav_speed_z', 'n_correct_z']
    use_cols   = [outcome] + predictors + ['subject', 'subj_sess']
    sub        = data[use_cols].dropna()
    n_trials   = len(sub)
    n_subj     = sub['subject'].nunique()
    n_sess     = sub['subj_sess'].nunique()

    formula = f'{outcome} ~ {main_pred} + nav_speed_z + n_correct_z'

    md  = smf.mixedlm(
        formula,
        sub,
        groups    = sub['subject'],
        vc_formula= {'subj_sess': '0 + C(subj_sess)'},
    )
    fit = md.fit(reml=True, method='lbfgs')

    print(f"\n{'='*65}")
    print(f"{label}  (N trials={n_trials}, subjects={n_subj}, sessions={n_sess})")
    print(f"{'='*65}")
    print(fit.summary())

    rows = []
    for pred in predictors:
        if pred in fit.params.index:
            rows.append({
                'label':     label,
                'outcome':   outcome,
                'predictor': pred,
                'beta':      fit.params[pred],
                'SE':        fit.bse[pred],
                'z':         fit.tvalues[pred],
                'p':         fit.pvalues[pred],
                'N_trials':  n_trials,
                'N_subj':    n_subj,
                'N_sess':    n_sess,
            })
    return pd.DataFrame(rows)

# ============================================================================
# 4. RUN 4 MODELS
# ============================================================================

res_A = run_lmm(
    outcome   = 'encoding_osc_ssl_LHP_z',
    main_pred = 'sp_clustering_LHP_z',
    label     = 'A: LHP gamma ~ SP + nav_speed + n_correct',
    data      = df,
)

res_B = run_lmm(
    outcome   = 'encoding_osc_ssl_RHP_z',
    main_pred = 'sp_clustering_RHP_z',
    label     = 'B: RHP gamma ~ SP + nav_speed + n_correct',
    data      = df,
)

res_C = run_lmm(
    outcome   = 'encoding_osc_ssl_LHP_z',
    main_pred = 'tc_clustering_LHP_z',
    label     = 'C: LHP gamma ~ TC + nav_speed + n_correct',
    data      = df,
)

res_D = run_lmm(
    outcome   = 'encoding_osc_ssl_RHP_z',
    main_pred = 'tc_clustering_RHP_z',
    label     = 'D: RHP gamma ~ TC + nav_speed + n_correct',
    data      = df,
)

# ============================================================================
# 5. COMBINE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

# BH-FDR across all 12 predictor tests (3 predictors × 4 models)
_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')

results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else (
              '†'   if p < .10  else '')))
)

# ============================================================================
# 6. PRINT SUMMARY
# ============================================================================

print("\n\n" + "=" * 75)
print(f"ALL RESULTS — BH-FDR corrected across {len(results)} tests")
print("=" * 75)
print(
    results[[
        'label', 'predictor', 'beta', 'SE', 'z', 'p', 'p_fdr', 'sig',
        'N_trials', 'N_subj',
    ]].to_string(index=False)
)

sig = results[results['sig'].isin(['*','**','***'])]
print(f"\n\nSIGNIFICANT (p_fdr < .05): {len(sig)} of {len(results)} tests")
if len(sig):
    print(sig[['label','predictor','beta','SE','z','p','p_fdr','sig']].to_string(index=False))
else:
    print("  None.")

# ============================================================================
# 7. SAVE
# ============================================================================

results.to_csv('gamma_osc_lmm_results.csv', index=False)
print("\nSaved: gamma_osc_lmm_results.csv")

Trial-level shape: (217, 22)
Subjects:  36
Sessions:  {'R1494D': 2, 'R1501J': 2, 'R1502D': 2, 'R1503E': 3, 'R1504E': 1, 'R1509E': 6, 'R1520T': 1, 'R1521E': 3, 'R1523E': 2, 'R1529T': 1, 'R1530J': 1, 'R1532T': 1, 'R1536J': 5, 'R1537T': 2, 'R1538E': 2, 'R1539E': 2, 'R1542J': 2, 'R1543E': 4, 'R1546E': 2, 'R1551T': 3, 'R1554T': 2, 'R1560T': 1, 'R1561E': 2, 'R1564J': 1, 'R1567T': 2, 'R1569T': 1, 'R1570T': 2, 'R1571T': 2, 'R1572T': 3, 'R1573T': 1, 'R1575E': 1, 'R1620J': 4, 'R1637T': 1, 'R1642J': 2, 'R1651J': 3, 'R1653J': 1}
Subject×session units: 76

Descriptives (z-scored):
       encoding_osc_ssl_LHP_z  encoding_osc_ssl_RHP_z  sp_clustering_LHP_z  sp_clustering_RHP_z  tc_clustering_LHP_z  tc_clustering_RHP_z  n_correct_z  nav_speed_z
count                 163.000                 161.000              163.000              161.000              163.000              161.000      217.000      217.000
mean                   -0.000                  -0.000                0.000                0.000  

In [7]:
"""
Interaction LMM (subject-level, long format):
  clustering_score ~ osc_power * clustering_type + nav_speed + n_correct
  + (1 | subject)

  - osc_power  = mean(LHP, RHP) oscillatory SSL, z-scored
  - clustering_type = SP vs TC (dummy-coded, SP=0, TC=1)
  - Two models: encoding power, retrieval power
  - Subject random intercept (each subject contributes 2 rows: SP and TC)

Figure: 2 panels side by side (encoding / retrieval)
  Each panel: grouped bar (beta ± SE) + interaction scatter inset
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD & SUBJECT-LEVEL AVERAGE
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'encoding_osc_ssl_LHP',  'encoding_osc_ssl_RHP',
    'retrieval_osc_ssl_LHP', 'retrieval_osc_ssl_RHP',
    'sp_clustering_LHP',     'sp_clustering_RHP',
    'tc_clustering_LHP',     'tc_clustering_RHP',
    'n_correct',             'nav_time_seconds',
]
cols = [c for c in cols if c in df.columns]
subj = df.groupby('subject')[cols].mean().reset_index()
subj['nav_speed'] = 1.0 / subj['nav_time_seconds']

# ── bilateral averages ───────────────────────────────────────────────────────
subj['osc_enc']  = subj[['encoding_osc_ssl_LHP',  'encoding_osc_ssl_RHP']].mean(axis=1)
subj['osc_ret']  = subj[['retrieval_osc_ssl_LHP', 'retrieval_osc_ssl_RHP']].mean(axis=1)
subj['sp_clust'] = subj[['sp_clustering_LHP',     'sp_clustering_RHP']].mean(axis=1)
subj['tc_clust'] = subj[['tc_clustering_LHP',     'tc_clustering_RHP']].mean(axis=1)

# ── z-score neural & covariate columns across subjects ──────────────────────
for col in ['osc_enc', 'osc_ret', 'n_correct', 'nav_speed']:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

print(f"Subjects with complete bilateral data: {subj[['osc_enc','osc_ret','sp_clust','tc_clust']].dropna().shape[0]}")

# ============================================================================
# 2. BUILD LONG FORMAT
#    Each subject → 2 rows: one for SP, one for TC
#    IMPORTANT: clustering scores are z-scored JOINTLY across both types
#    so that type_dummy and the interaction are estimable.
# ============================================================================

rows = []
for _, s in subj.iterrows():
    for ctype, clust_raw in [('SP', s['sp_clust']), ('TC', s['tc_clust'])]:
        rows.append({
            'subject':        s['subject'],
            'clust_score_raw': clust_raw,
            'clust_type':     ctype,
            'osc_enc_z':      s['osc_enc_z'],
            'osc_ret_z':      s['osc_ret_z'],
            'nav_speed_z':    s['nav_speed_z'],
            'n_correct_z':    s['n_correct_z'],
        })

long = pd.DataFrame(rows).dropna()
long['type_dummy'] = (long['clust_type'] == 'TC').astype(float)   # SP=0, TC=1

# Joint z-score: preserves mean difference between SP and TC
long['clust_score_z'] = (
    (long['clust_score_raw'] - long['clust_score_raw'].mean())
    / long['clust_score_raw'].std()
)

print(f"\nLong-format shape: {long.shape}")
print(long.groupby('clust_type')['clust_score_z'].describe().round(3))

# ============================================================================
# 3. FIT LMMs
# ============================================================================

def fit_lmm(power_col, stage_label):
    """
    clustering_score ~ power * type + nav_speed + n_correct + (1|subject)
    Returns fitted model + tidy results dataframe.
    """
    formula = (
        f'clust_score_z ~ {power_col} * type_dummy '
        f'+ nav_speed_z + n_correct_z'
    )
    md  = smf.mixedlm(formula, long, groups=long['subject'])
    fit = md.fit(reml=False, method='powell')

    print(f"\n{'='*65}")
    print(f"MODEL: {stage_label}")
    print(f"Formula: {formula}")
    print(f"{'='*65}")
    print(fit.summary())

    terms = [power_col, 'type_dummy', f'{power_col}:type_dummy',
             'nav_speed_z', 'n_correct_z']
    rows_out = []
    for term in terms:
        if term in fit.params.index:
            rows_out.append({
                'stage':     stage_label,
                'term':      term,
                'beta':      fit.params[term],
                'SE':        fit.bse[term],
                't':         fit.tvalues[term],
                'p':         fit.pvalues[term],
            })
    return fit, pd.DataFrame(rows_out)

fit_enc, res_enc = fit_lmm('osc_enc_z', 'Encoding')
fit_ret, res_ret = fit_lmm('osc_ret_z', 'Retrieval')

# ── FDR across all tests ─────────────────────────────────────────────────────
all_res = pd.concat([res_enc, res_ret], ignore_index=True)
_, all_res['p_fdr'], _, _ = multipletests(all_res['p'].values, method='fdr_bh')
all_res['sig'] = all_res['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else (
              '†' if p < .10 else ''))))

print("\n\n" + "="*65)
print("ALL TERMS — BH-FDR corrected")
print("="*65)
print(all_res[['stage','term','beta','SE','t','p','p_fdr','sig']].to_string(index=False))

all_res.to_csv('lmm_interaction_results.csv', index=False)
print("\nSaved: lmm_interaction_results.csv")

# ============================================================================
# 4. PLOTTING
# ============================================================================

BLUE   = '#185FA5'   # SP
GREEN  = '#0F6E56'   # TC
GRAY   = '#888780'
RED    = '#D85A30'
PURPLE = '#7B2D8B'   # interaction term

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'xtick.major.size':  3,
    'ytick.major.size':  3,
})

TERM_LABELS = {
    'osc_enc_z':              'Osc power',
    'osc_ret_z':              'Osc power',
    'type_dummy':             'Type\n(TC vs SP)',
    'osc_enc_z:type_dummy':   'Power ×\nType',
    'osc_ret_z:type_dummy':   'Power ×\nType',
    'nav_speed_z':            'Nav speed',
    'n_correct_z':            'N correct',
}
TERM_COLORS = {
    'osc_enc_z':              BLUE,
    'osc_ret_z':              BLUE,
    'type_dummy':             GREEN,
    'osc_enc_z:type_dummy':   PURPLE,
    'osc_ret_z:type_dummy':   PURPLE,
    'nav_speed_z':            GRAY,
    'n_correct_z':            GRAY,
}

def add_sig(ax, x, y_top, sig, fontsize=10):
    if sig in ('*','**','***'):
        ax.text(x, y_top + 0.04, sig, ha='center', va='bottom',
                fontsize=fontsize, color=RED, fontweight='bold')
    elif sig == '†':
        ax.text(x, y_top + 0.04, '†', ha='center', va='bottom',
                fontsize=fontsize+1, color=GRAY)

# ── Figure: 2 panels (encoding / retrieval) each with bar + scatter ──────────

fig = plt.figure(figsize=(16, 11))
outer_gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38)

stage_configs = [
    ('Encoding', 'osc_enc_z',
     ['osc_enc_z', 'type_dummy', 'osc_enc_z:type_dummy', 'nav_speed_z', 'n_correct_z']),
    ('Retrieval', 'osc_ret_z',
     ['osc_ret_z', 'type_dummy', 'osc_ret_z:type_dummy', 'nav_speed_z', 'n_correct_z']),
]

for col_idx, (stage, power_col, term_order) in enumerate(stage_configs):

    inner_gs = gridspec.GridSpecFromSubplotSpec(
        2, 1, subplot_spec=outer_gs[col_idx],
        height_ratios=[1.1, 1], hspace=0.52
    )

    stage_res = all_res[all_res['stage'] == stage].set_index('term')

    # ── TOP: bar plot of all 5 terms ─────────────────────────────────────
    ax_bar = fig.add_subplot(inner_gs[0])

    x_pos   = np.arange(len(term_order))
    betas   = [stage_res.loc[t, 'beta'] if t in stage_res.index else 0 for t in term_order]
    ses     = [stage_res.loc[t, 'SE']   if t in stage_res.index else 0 for t in term_order]
    sigs    = [stage_res.loc[t, 'sig']  if t in stage_res.index else '' for t in term_order]
    colors  = [TERM_COLORS.get(t, GRAY) for t in term_order]
    xlabels = [TERM_LABELS.get(t, t)    for t in term_order]

    bars = ax_bar.bar(x_pos, betas, 0.55, color=colors, alpha=0.85,
                      zorder=3, linewidth=0)
    ax_bar.errorbar(x_pos, betas, yerr=ses,
                    fmt='none', color='#444', linewidth=1.2, capsize=3, zorder=4)

    for xi, (b, se, sig) in enumerate(zip(betas, ses, sigs)):
        ytop = b + se if b >= 0 else b - se
        add_sig(ax_bar, xi, ytop if b >= 0 else ytop - 0.08, sig)

    ax_bar.axhline(0, color='#aaa', linewidth=0.8, zorder=2)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(xlabels, fontsize=9.5)
    ax_bar.set_ylabel('Standardized β', fontsize=10)
    ax_bar.set_title(
        f'{"A" if col_idx==0 else "B"}   {stage} — '
        f'Clustering ~ Osc power × Type',
        fontsize=11, fontweight='normal', pad=8
    )
    ymin = min(min(betas) - max(ses) - 0.25, -0.4)
    ymax = max(max(betas) + max(ses) + 0.25,  0.7)
    ax_bar.set_ylim(ymin, ymax)
    ax_bar.yaxis.grid(True, linewidth=0.4, color='#ddd', zorder=0)
    ax_bar.set_axisbelow(True)

    # colour-coded x-tick labels
    for tick_lbl, term in zip(ax_bar.get_xticklabels(), term_order):
        tick_lbl.set_color(TERM_COLORS.get(term, GRAY))

    # legend patch
    import matplotlib.patches as mpatches
    leg_els = [
        mpatches.Patch(color=BLUE,   label='Osc power (bilateral)'),
        mpatches.Patch(color=GREEN,  label='Clustering type (TC vs SP)'),
        mpatches.Patch(color=PURPLE, label='Power × Type interaction'),
    ]
    ax_bar.legend(handles=leg_els, fontsize=8, frameon=False,
                  loc='upper right', bbox_to_anchor=(1.02, 1.02))

    # ── BOTTOM: interaction scatter ───────────────────────────────────────
    ax_sc = fig.add_subplot(inner_gs[1])

    # SP and TC separately, same x-axis (bilateral osc power)
    for ctype, color, ls in [('SP', BLUE, '-'), ('TC', GREEN, '--')]:
        sub = long[long['clust_type'] == ctype][[power_col, 'clust_score_z']].dropna()
        x_  = sub[power_col].values
        y_  = sub['clust_score_z'].values

        slope, intercept, r, p_r, _ = stats.linregress(x_, y_)
        xline = np.linspace(x_.min(), x_.max(), 100)

        ax_sc.scatter(x_, y_, color=color, alpha=0.65, s=38, zorder=3,
                      edgecolors='white', linewidths=0.4, label=ctype)
        ax_sc.plot(xline, slope * xline + intercept,
                   color=color, lw=1.8, ls=ls, zorder=4)

        # CI band
        n   = len(x_)
        se_ = np.std(y_ - (slope * x_ + intercept), ddof=2)
        ci  = se_ * stats.t.ppf(0.975, df=n-2) * np.sqrt(
                  1/n + (xline - x_.mean())**2 / np.sum((x_ - x_.mean())**2))
        ax_sc.fill_between(xline,
                           slope*xline + intercept - ci,
                           slope*xline + intercept + ci,
                           color=color, alpha=0.10, zorder=2)

        p_str = f'p = {p_r:.3f}' if p_r >= 0.001 else 'p < 0.001'
        y_ann = 0.93 if ctype == 'SP' else 0.82
        ax_sc.text(0.05, y_ann, f'{ctype}  r = {r:.2f},  {p_str}',
                   transform=ax_sc.transAxes, fontsize=8.5,
                   color=color, va='top')

    # annotate interaction p_fdr
    ix_term = f'{power_col}:type_dummy'
    if ix_term in stage_res.index:
        ix_sig = stage_res.loc[ix_term, 'sig']
        ix_p   = stage_res.loc[ix_term, 'p_fdr']
        ix_str = f'Interaction p_fdr = {ix_p:.3f}'
        if ix_sig:
            ix_str += f'  {ix_sig}'
        ax_sc.text(0.05, 0.71, ix_str,
                   transform=ax_sc.transAxes, fontsize=8,
                   color=PURPLE, va='top', style='italic')

    ax_sc.axhline(0, color='#ccc', lw=0.6)
    ax_sc.axvline(0, color='#ccc', lw=0.6)
    ax_sc.set_xlabel(f'Bilateral osc power — {stage.lower()} (z)', fontsize=10)
    ax_sc.set_ylabel('Clustering score (z)', fontsize=10)
    ax_sc.set_title(
        f'{"C" if col_idx==0 else "D"}   SP vs TC clustering — {stage.lower()} power',
        fontsize=11, fontweight='normal', pad=8
    )
    ax_sc.legend(fontsize=9, frameon=False)
    ax_sc.yaxis.grid(True, linewidth=0.4, color='#ddd', zorder=0)
    ax_sc.set_axisbelow(True)

fig.suptitle(
    'Clustering score ~ Oscillatory power × Clustering type\n'
    '(LMM, subject-level, bilateral HC average)',
    fontsize=13, fontweight='bold', y=1.02
)
fig.text(
    0.5, -0.01,
    '* p < .05   ** p < .01   *** p < .001 (FDR-corrected)   '
    '† p < .10 trend   Error bars = ±1 SE\n'
    'Blue = SP clustering,  Green = TC clustering,  Purple = interaction term',
    ha='center', fontsize=8, color='#888'
)

fig.savefig('figure_interaction_osc_clustering.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close(fig)
print("\nSaved: figure_interaction_osc_clustering.png")

Subjects with complete bilateral data: 36

Long-format shape: (72, 9)
            count   mean    std    min    25%    50%    75%    max
clust_type                                                        
SP           36.0 -0.304  1.069 -3.041 -0.821 -0.387  0.253  1.916
TC           36.0  0.304  0.835 -1.665 -0.296  0.295  0.781  1.782

MODEL: Encoding
Formula: clust_score_z ~ osc_enc_z * type_dummy + nav_speed_z + n_correct_z
             Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  clust_score_z
No. Observations:   72       Method:              ML           
No. Groups:         36       Scale:               0.7174       
Min. group size:    2        Log-Likelihood:      -93.1844     
Max. group size:    2        Converged:           Yes          
Mean group size:    2.0                                        
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------